A notebook for the quick visualization of already stitched object reconstructions.

In [1]:
import cmcrameri.cm as cmc
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

%matplotlib inline

### Mass plot of many objects with ground truth

In [3]:
# Load all inference results in results_dir
from pathlib import Path

results_dir = '/scratch/aileenluo/ptycho-vit/eps/results/params_tokens/300M_5'
results_path = Path(results_dir)
object_names = sorted(
    gt_path.name.removeprefix('gt_object_').removesuffix('.npy')
    for gt_path in results_path.glob('gt_object_*.npy')
)

missing = []
gt_objects = {}
pred_amp_objects = {}
pred_ph_objects = {}

for name in object_names:
    gt_path = results_path / f'gt_object_{name}.npy'
    pred_amp_path = results_path / f'pred_amp_object_{name}.npy'
    pred_ph_path = results_path / f'pred_ph_object_{name}.npy'

    if not pred_amp_path.exists() or not pred_ph_path.exists():
        missing.append(name)
        continue

    gt_objects[name] = np.load(gt_path)
    pred_amp_objects[name] = np.load(pred_amp_path)
    pred_ph_objects[name] = np.load(pred_ph_path)

loaded_object_names = list(gt_objects)

if missing:
    print(f'Skipped {len(missing)} objects with missing prediction files: {missing}')

len(loaded_object_names), {name: gt_objects[name].shape for name in loaded_object_names[:5]}

(10,
 {'n07581931_10626': (350, 350),
  'n07581931_10642': (350, 350),
  'n07581931_1075': (350, 350),
  'n07581931_10953': (350, 350),
  'n07581931_11344': (350, 350)})

In [ ]:
def _add_colorbar(fig, axis, image):
    divider = make_axes_locatable(axis)
    cax = divider.append_axes('right', size='5%', pad=0.05)
    fig.colorbar(image, cax=cax, orientation='vertical')


def _centered_range(values, target_range):
    center = 0.5 * (np.nanmin(values) + np.nanmax(values))
    half_range = 0.5 * target_range
    return center - half_range, center + half_range


def plot_loaded_objects(object_names=None, max_objects=None):
    if object_names is None:
        object_names = loaded_object_names
    if max_objects is not None:
        object_names = object_names[:max_objects]

    figures = []
    for name in object_names:
        gt_object = gt_objects[name]
        pred_amp_object = pred_amp_objects[name]
        pred_ph_object = pred_ph_objects[name]

        gt_amp = np.abs(gt_object)
        gt_ph = np.angle(gt_object)

        amp_vmin, amp_vmax = np.nanmin(gt_amp), np.nanmax(gt_amp)
        ph_vmin, ph_vmax = np.nanmin(gt_ph), np.nanmax(gt_ph)
        pred_ph_vmin, pred_ph_vmax = _centered_range(pred_ph_object, ph_vmax - ph_vmin)

        f, ax = plt.subplots(figsize=(9, 8), ncols=2, nrows=2)
        f.suptitle(name)

        gt0 = ax[0, 0].imshow(gt_amp, interpolation='none', vmin=amp_vmin, vmax=amp_vmax, cmap=cmc.oslo)
        _add_colorbar(f, ax[0, 0], gt0)
        ax[0, 0].set_title('GT amplitude')

        gt1 = ax[0, 1].imshow(gt_ph, interpolation='none', vmin=ph_vmin, vmax=ph_vmax, cmap=cmc.vik)
        _add_colorbar(f, ax[0, 1], gt1)
        ax[0, 1].set_title('GT phase')

        pred0 = ax[1, 0].imshow(pred_amp_object, interpolation='none', vmin=amp_vmin, vmax=amp_vmax, cmap=cmc.oslo)
        _add_colorbar(f, ax[1, 0], pred0)
        ax[1, 0].set_title('Predicted amplitude')

        pred1 = ax[1, 1].imshow(pred_ph_object, interpolation='none', vmin=pred_ph_vmin, vmax=pred_ph_vmax, cmap=cmc.vik)
        _add_colorbar(f, ax[1, 1], pred1)
        ax[1, 1].set_title('Predicted phase')

        plt.tight_layout()
        figures.append(f)

    return figures


# Plot every loaded object. Use max_objects to preview a subset first.
figures = plot_loaded_objects()

In [ ]:
# Plot scaling values from workspace CSV, excluding the 1B column
scaling_df = pd.read_csv('workspace/scaling_plot_values_complete.csv')

plot_df = scaling_df.drop(columns=['1B'])
value_columns = [col for col in plot_df.columns if col != 'tokens']
colors = plt.cm.viridis(np.linspace(0, 1, len(value_columns)))
ax = plot_df.plot(x='tokens', y=value_columns, marker='o', figsize=(8, 5), color=colors)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Tokens')
ax.set_ylabel('Val. loss')
ax.set_title('Scaling Plot Values')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()

In [ ]:
# Compute and plot SSIM/PSNR/PCC scaling values from full-object inference results
import re
from pathlib import Path

from metrics import compute_psnr, compute_ssim, prepare_image_metric_inputs
from scipy.stats import pearsonr

ssim_results_root = Path('/scratch/aileenluo/ptycho-vit/eps/results/params_tokens')
scaling_df = pd.read_csv('workspace/scaling_plot_values_complete.csv')


def _model_size_key(model_size):
    match = re.fullmatch(r'(\d+)([KMB])', model_size)
    if match is None:
        return model_size
    value, suffix = match.groups()
    multiplier = {'K': 1e3, 'M': 1e6, 'B': 1e9}[suffix]
    return int(value) * multiplier


def _load_result_objects(result_path):
    object_names = sorted(
        gt_path.name.removeprefix('gt_object_').removesuffix('.npy')
        for gt_path in result_path.glob('gt_object_*.npy')
    )

    objects = []
    missing = []
    for name in object_names:
        gt_path = result_path / f'gt_object_{name}.npy'
        pred_amp_path = result_path / f'pred_amp_object_{name}.npy'
        pred_ph_path = result_path / f'pred_ph_object_{name}.npy'

        if not pred_amp_path.exists() or not pred_ph_path.exists():
            missing.append(name)
            continue

        objects.append((name, np.load(gt_path), np.load(pred_amp_path), np.load(pred_ph_path)))

    return objects, missing


def _finite_pair_values(pred_values, gt_values):
    pred_flat = np.asarray(pred_values).ravel()
    gt_flat = np.asarray(gt_values).ravel()
    valid = np.isfinite(pred_flat) & np.isfinite(gt_flat)
    return pred_flat[valid], gt_flat[valid]


def _compute_result_dir_ssim(result_path):
    objects, missing = _load_result_objects(result_path)
    amp_ssims = []
    phase_ssims = []
    amp_psnrs = []
    phase_psnrs = []
    amp_pccs = []
    phase_pccs = []

    for _, gt_object, pred_amp_object, pred_ph_object in objects:
        gt_amp = np.abs(gt_object)
        gt_phase = np.angle(gt_object)
        amp_pred, amp_target, amp_data_range = prepare_image_metric_inputs(
            pred_amp_object, gt_amp
        )
        phase_pred, phase_target, phase_data_range = prepare_image_metric_inputs(
            pred_ph_object, gt_phase, align_mean=True
        )

        pred_amp_values, gt_amp_values = _finite_pair_values(pred_amp_object, gt_amp)
        pred_phase_values, gt_phase_values = _finite_pair_values(pred_ph_object, gt_phase)
        amp_pccs.append(pearsonr(pred_amp_values, gt_amp_values).statistic if pred_amp_values.size >= 2 else np.nan)
        phase_pccs.append(pearsonr(pred_phase_values, gt_phase_values).statistic if pred_phase_values.size >= 2 else np.nan)

        amp_ssims.append(compute_ssim(amp_pred, amp_target, data_range=amp_data_range))
        phase_ssims.append(compute_ssim(phase_pred, phase_target, data_range=phase_data_range))
        amp_psnrs.append(compute_psnr(amp_pred, amp_target, data_range=amp_data_range))
        phase_psnrs.append(compute_psnr(phase_pred, phase_target, data_range=phase_data_range))

    return {
        'n_objects': len(objects),
        'n_missing': len(missing),
        'amp_ssim': np.nanmean(amp_ssims) if amp_ssims else np.nan,
        'phase_ssim': np.nanmean(phase_ssims) if phase_ssims else np.nan,
        'amp_psnr': np.nanmean(amp_psnrs) if amp_psnrs else np.nan,
        'phase_psnr': np.nanmean(phase_psnrs) if phase_psnrs else np.nan,
        'amp_pcc': np.nanmean(amp_pccs) if amp_pccs else np.nan,
        'phase_pcc': np.nanmean(phase_pccs) if phase_pccs else np.nan,
    }


result_dir_pattern = re.compile(r'^(?P<model_size>\d+[KMB])_(?P<data_percent>\d+)$')
records = []

for result_path in sorted(ssim_results_root.iterdir()):
    if not result_path.is_dir():
        continue

    match = result_dir_pattern.fullmatch(result_path.name)
    if match is None:
        continue

    model_size = match.group('model_size')
    data_percent = int(match.group('data_percent'))
    result = _compute_result_dir_ssim(result_path)
    records.append({'model_size': model_size, 'data_percent': data_percent, **result})

ssim_df = pd.DataFrame(records).sort_values(
    ['data_percent', 'model_size'],
    key=lambda col: col.map(_model_size_key) if col.name == 'model_size' else col,
)

data_percent_to_tokens = dict(zip(sorted(ssim_df['data_percent'].unique()), scaling_df['tokens'].to_numpy()))
ssim_df['tokens'] = ssim_df['data_percent'].map(data_percent_to_tokens)

model_sizes = sorted(ssim_df['model_size'].unique(), key=_model_size_key)
amp_ssim_df = ssim_df.pivot(index='tokens', columns='model_size', values='amp_ssim').reindex(columns=model_sizes)
phase_ssim_df = ssim_df.pivot(index='tokens', columns='model_size', values='phase_ssim').reindex(columns=model_sizes)
amp_psnr_df = ssim_df.pivot(index='tokens', columns='model_size', values='amp_psnr').reindex(columns=model_sizes)
phase_psnr_df = ssim_df.pivot(index='tokens', columns='model_size', values='phase_psnr').reindex(columns=model_sizes)
amp_pcc_df = ssim_df.pivot(index='tokens', columns='model_size', values='amp_pcc').reindex(columns=model_sizes)
phase_pcc_df = ssim_df.pivot(index='tokens', columns='model_size', values='phase_pcc').reindex(columns=model_sizes)

colors = plt.cm.viridis(np.linspace(0, 1, len(model_sizes)))
fig, ax = plt.subplots(ncols=2, figsize=(12, 5), sharex=True)

amp_ssim_df.plot(marker='o', color=colors, ax=ax[0])
phase_ssim_df.plot(marker='o', color=colors, ax=ax[1])

for axis, title in zip(ax, ['Amplitude SSIM', 'Phase SSIM']):
    axis.set_xscale('log')
    axis.set_xlabel('Tokens')
    axis.set_ylabel('SSIM')
    axis.set_title(title)
    axis.grid(True, which='both', alpha=0.3)
    axis.legend(title='Model size')

plt.tight_layout()

fig, ax = plt.subplots(ncols=2, figsize=(12, 5), sharex=True)

amp_psnr_df.plot(marker='o', color=colors, ax=ax[0])
phase_psnr_df.plot(marker='o', color=colors, ax=ax[1])

for axis, title in zip(ax, ['Amplitude PSNR', 'Phase PSNR']):
    axis.set_xscale('log')
    axis.set_xlabel('Tokens')
    axis.set_ylabel('PSNR (dB)')
    axis.set_title(title)
    axis.grid(True, which='both', alpha=0.3)
    axis.legend(title='Model size')

plt.tight_layout()

fig, ax = plt.subplots(ncols=2, figsize=(12, 5), sharex=True)

amp_pcc_df.plot(marker='o', color=colors, ax=ax[0])
phase_pcc_df.plot(marker='o', color=colors, ax=ax[1])

for axis, title in zip(ax, ['Amplitude PCC', 'Phase PCC']):
    axis.set_xscale('log')
    axis.set_xlabel('Tokens')
    axis.set_ylabel('Pearson correlation coefficient')
    axis.set_title(title)
    axis.grid(True, which='both', alpha=0.3)
    axis.legend(title='Model size')

plt.tight_layout()

ssim_df

In [ ]:
# Plot SSIM and PCC scaling values, excluding 1% training data
non_1_percent_df = ssim_df[ssim_df['data_percent'] != 1]

amp_ssim_non_1_df = non_1_percent_df.pivot(index='tokens', columns='model_size', values='amp_ssim').reindex(columns=model_sizes)
phase_ssim_non_1_df = non_1_percent_df.pivot(index='tokens', columns='model_size', values='phase_ssim').reindex(columns=model_sizes)
amp_pcc_non_1_df = non_1_percent_df.pivot(index='tokens', columns='model_size', values='amp_pcc').reindex(columns=model_sizes)
phase_pcc_non_1_df = non_1_percent_df.pivot(index='tokens', columns='model_size', values='phase_pcc').reindex(columns=model_sizes)

colors = plt.cm.viridis(np.linspace(0, 1, len(model_sizes)))
fig, ax = plt.subplots(ncols=2, figsize=(12, 5), sharex=True)

amp_ssim_non_1_df.plot(marker='o', color=colors, ax=ax[0])
phase_ssim_non_1_df.plot(marker='o', color=colors, ax=ax[1])

for axis, title in zip(ax, ['Amplitude SSIM', 'Phase SSIM']):
    axis.set_xscale('log')
    axis.set_xlabel('Tokens')
    axis.set_ylabel('SSIM')
    axis.set_title(f'{title}: excluding 1% training data')
    axis.grid(True, which='both', alpha=0.3)
    axis.legend(title='Model size')

plt.tight_layout()

fig, ax = plt.subplots(ncols=2, figsize=(12, 5), sharex=True)

amp_pcc_non_1_df.plot(marker='o', color=colors, ax=ax[0])
phase_pcc_non_1_df.plot(marker='o', color=colors, ax=ax[1])

for axis, title in zip(ax, ['Amplitude PCC', 'Phase PCC']):
    axis.set_xscale('log')
    axis.set_xlabel('Tokens')
    axis.set_ylabel('Pearson correlation coefficient')
    axis.set_title(f'{title}: excluding 1% training data')
    axis.grid(True, which='both', alpha=0.3)
    axis.legend(title='Model size')

plt.tight_layout()

## Crop Size Series Evaluation

Load flyscan outputs saved with `_crop<N>` in the filename, visualize the crop-size series, and compute SSIM/PCC per crop size. This section is self-contained except for the top import/style cell.

In [ ]:
# Load and plot the crop-size series
import re
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt

# Set this to the directory containing files like:
#   pred_amp_object_<object>_crop104.npy
#   pred_ph_object_<object>_crop104.npy
# Ground truth files named gt_object_<object>_crop104.npy are optional.
crop_results_dir = Path('/mnt/localdisk/scratch/aileenluo/results/run164')

# Set to a specific object name to plot it, or leave as None to use the first loaded object.
crop_series_object_name = 'scan_267_100259'
crop_series_max_plot_crops = 7


def _add_colorbar_crop_series(fig, axis, image):
    divider = make_axes_locatable(axis)
    cax = divider.append_axes('right', size='5%', pad=0.05)
    fig.colorbar(image, cax=cax, orientation='vertical')


def load_crop_series(result_dir):
    result_dir = Path(result_dir)
    crop_pattern = re.compile(r'^pred_amp_object_(?P<object>.+)_crop(?P<crop>\d+)\.npy$')
    entries = []
    missing_predictions = []

    # Discover the series from predictions so ground truth is not required for plotting.
    for pred_amp_path in sorted(result_dir.glob('pred_amp_object_*_crop*.npy')):
        match = crop_pattern.fullmatch(pred_amp_path.name)
        if match is None:
            continue

        object_name = match.group('object')
        crop_size = int(match.group('crop'))
        pred_ph_path = result_dir / f'pred_ph_object_{object_name}_crop{crop_size}.npy'
        gt_path = result_dir / f'gt_object_{object_name}_crop{crop_size}.npy'

        if not pred_ph_path.exists():
            missing_predictions.append({
                'object_name': object_name,
                'crop_size': crop_size,
                'pred_amp_path': pred_amp_path,
                'pred_ph_path': pred_ph_path,
            })
            continue

        entries.append({
            'object_name': object_name,
            'crop_size': crop_size,
            'pred_amp_path': pred_amp_path,
            'pred_ph_path': pred_ph_path,
            'gt_path': gt_path if gt_path.exists() else None,
        })

    entries = sorted(entries, key=lambda item: (item['object_name'], item['crop_size']))
    return entries, missing_predictions


def load_crop_predictions(entry):
    return {
        'pred_amp_object': np.load(entry['pred_amp_path']),
        'pred_ph_object': np.load(entry['pred_ph_path']),
    }


def plot_crop_series_objects(entries, object_name=None, max_crops=8):
    if not entries:
        print('No crop-series objects to plot.')
        return None

    available_objects = sorted({entry['object_name'] for entry in entries})
    if object_name is None:
        object_name = available_objects[0]
    if object_name not in available_objects:
        raise ValueError(f'Object {object_name!r} not found. Available examples: {available_objects[:5]}')

    selected_entries = [entry for entry in entries if entry['object_name'] == object_name]
    selected_entries = sorted(selected_entries, key=lambda item: item['crop_size'])
    if max_crops is not None:
        selected_entries = selected_entries[:max_crops]

    n_crops = len(selected_entries)
    fig, ax = plt.subplots(
        nrows=2,
        ncols=n_crops,
        figsize=(3.5 * n_crops, 7),
        squeeze=False,
        dpi=300,
    )
    fig.suptitle(f'Crop series: {object_name}')

    for col, entry in enumerate(selected_entries):
        loaded = load_crop_predictions(entry)
        images = [
            (loaded['pred_amp_object'], 'Amplitude', cmc.oslo, 0.6, 0.8),
            (loaded['pred_ph_object'], 'Phase', cmc.grayC, -1, 1),
        ]

        for row, (image, label, cmap, vmin, vmax) in enumerate(images):
            image_artist = ax[row, col].imshow(
                image,
                interpolation='none',
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
                origin='lower',
            )
            ax[row, col].set_title(f'{label}, crop {entry["crop_size"]}')
            ax[row, col].set_xticks([])
            ax[row, col].set_yticks([])
            _add_colorbar_crop_series(fig, ax[row, col], image_artist)

    plt.tight_layout(rect=(0, 0, 1, 0.95))
    return fig


crop_series_entries, crop_series_missing_predictions = load_crop_series(crop_results_dir)
n_with_ground_truth = sum(entry['gt_path'] is not None for entry in crop_series_entries)
print(
    f'Loaded metadata for {len(crop_series_entries)} crop-series outputs from {crop_results_dir} '
    f'({n_with_ground_truth} with ground truth)'
)
if crop_series_missing_predictions:
    print(f'Skipped {len(crop_series_missing_predictions)} entries with missing phase predictions')

crop_objects_fig = plot_crop_series_objects(
    crop_series_entries,
    object_name=crop_series_object_name,
    max_crops=crop_series_max_plot_crops,
)

In [ ]:
# Calculate crop-series metrics only where reference ground truth is available
import pandas as pd


def _crop_gt_object_for_crop_size(gt_object, crop_size):
    crop_slice = slice(crop_size + 50, -crop_size - 51)
    return gt_object[crop_slice, crop_slice]


def _require_same_shape(pred_values, gt_values, label, entry):
    if pred_values.shape != gt_values.shape:
        raise ValueError(
            f"{label} shape mismatch for {entry['object_name']} crop {entry['crop_size']}: "
            f"prediction {pred_values.shape}, GT {gt_values.shape}"
        )


def _finite_pair_values(pred_values, gt_values):
    pred_flat = np.asarray(pred_values).ravel()
    gt_flat = np.asarray(gt_values).ravel()
    valid = np.isfinite(pred_flat) & np.isfinite(gt_flat)
    return pred_flat[valid], gt_flat[valid]


def _pcc(pred_values, gt_values):
    pred_values, gt_values = _finite_pair_values(pred_values, gt_values)
    if pred_values.size < 2:
        return np.nan
    return pearsonr(pred_values, gt_values).statistic


def compute_crop_series_metrics(entries):
    records = []

    for entry in entries:
        if entry['gt_path'] is None:
            continue

        loaded = load_crop_predictions(entry)
        gt_object = np.load(entry['gt_path'])
        crop_size = entry['crop_size']
        gt_object = _crop_gt_object_for_crop_size(gt_object, crop_size)
        pred_amp_object = np.flipud(loaded['pred_amp_object'])
        pred_ph_object = np.flipud(loaded['pred_ph_object'])

        gt_amp = np.abs(gt_object)
        gt_phase = np.angle(gt_object)
        _require_same_shape(pred_amp_object, gt_amp, 'Amplitude', entry)
        _require_same_shape(pred_ph_object, gt_phase, 'Phase', entry)
        amp_pred, amp_target, amp_data_range = prepare_image_metric_inputs(
            pred_amp_object, gt_amp
        )
        phase_pred, phase_target, phase_data_range = prepare_image_metric_inputs(
            pred_ph_object, gt_phase, align_mean=True
        )

        records.append({
            'object_name': entry['object_name'],
            'crop_size': entry['crop_size'],
            'amp_ssim': compute_ssim(amp_pred, amp_target, data_range=amp_data_range),
            'phase_ssim': compute_ssim(phase_pred, phase_target, data_range=phase_data_range),
            'amp_psnr': compute_psnr(amp_pred, amp_target, data_range=amp_data_range),
            'phase_psnr': compute_psnr(phase_pred, phase_target, data_range=phase_data_range),
            'amp_pcc': _pcc(pred_amp_object, gt_amp),
            'phase_pcc': _pcc(pred_ph_object, gt_phase),
            'pred_shape': pred_amp_object.shape,
            'gt_crop_shape': gt_amp.shape,
        })

    metrics_df = pd.DataFrame(records)
    if metrics_df.empty:
        return metrics_df, metrics_df

    summary_df = (
        metrics_df
        .groupby('crop_size', as_index=False)
        .agg(
            n_objects=('object_name', 'nunique'),
            amp_ssim=('amp_ssim', 'mean'),
            phase_ssim=('phase_ssim', 'mean'),
            amp_psnr=('amp_psnr', 'mean'),
            phase_psnr=('phase_psnr', 'mean'),
            amp_pcc=('amp_pcc', 'mean'),
            phase_pcc=('phase_pcc', 'mean'),
        )
        .sort_values('crop_size')
    )
    return metrics_df, summary_df


def plot_crop_series_metrics(summary_df):
    if summary_df.empty:
        print('No crop-series metrics to plot.')
        return None

    fig, ax = plt.subplots(ncols=2, nrows=3, figsize=(12, 13), sharex=True)
    metric_specs = [
        ('amp_ssim', 'Amplitude SSIM'),
        ('phase_ssim', 'Phase SSIM'),
        ('amp_psnr', 'Amplitude PSNR'),
        ('phase_psnr', 'Phase PSNR'),
        ('amp_pcc', 'Amplitude PCC'),
        ('phase_pcc', 'Phase PCC'),
    ]

    for axis, (column, title) in zip(ax.ravel(), metric_specs):
        axis.plot(summary_df['crop_size'], summary_df[column], marker='o')
        axis.set_title(title)
        axis.set_xlabel('Crop size')
        axis.set_ylabel(column)
        axis.grid(True, alpha=0.3)

    plt.tight_layout()
    return fig


n_without_ground_truth = sum(entry['gt_path'] is None for entry in crop_series_entries)
if n_without_ground_truth:
    print(f'Skipping metrics for {n_without_ground_truth} crop-series outputs without ground truth')

crop_metrics_df, crop_summary_df = compute_crop_series_metrics(crop_series_entries)
display(crop_summary_df)
display(crop_metrics_df)

crop_metrics_fig = plot_crop_series_metrics(crop_summary_df)